This script fetches hourly air quality data (AQI, PM10, PM2.5, NO2, CO, O3, SO2, CO2) from the Open-Meteo Air Quality API for multiple monitoring stations (2021–2025). Station metadata (coordinates, names) is read from a CSV file. Results for all locations are combined and saved as a single hourly CSV file.

Developed with the assistance of [Gemini](https://gemini.google.com)

In [4]:
import openmeteo_requests
import polars as pl
import requests_cache
import os
from retry_requests import retry
from datetime import datetime, timedelta, timezone

BASE_DIR   = "../../data"

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


# Get location information
locations_df = pl.read_csv(os.path.join(BASE_DIR, "BASt", "AggregatedDataForKiel", "meta_data_used_stations.csv"))

location_Zst = locations_df["Zst"].to_list()
location_names = locations_df["name"].to_list()
latitudes = locations_df["lat"].to_list()
longitudes = locations_df["lon"].to_list()

# URL and parameters remain the same
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": latitudes,
    "longitude": longitudes,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "hourly": ["european_aqi", "pm10", "pm2_5", "nitrogen_dioxide", "carbon_monoxide", "ozone", "sulphur_dioxide", "carbon_dioxide"]
}
responses = openmeteo.weather_api(url, params=params)

In [5]:
# Hourly Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "AirQuality", "air_quality_hourly.csv")

all_dataframes = []
# Process first location
for i, response in enumerate(responses):
    # Process location
    response = responses[0]

    # Get name
    zst = location_Zst[i]
    name = location_names[i]
    
    # --- Process Hourly Data ---
    hourly = response.Hourly()
    
    #  calculate the range using datetime objects
    start_h = datetime.fromtimestamp(hourly.Time(), tz=timezone.utc)
    end_h = datetime.fromtimestamp(hourly.TimeEnd(), tz=timezone.utc)
    
    interval_h = timedelta(seconds=hourly.Interval())
    
    hourly_dataframe = pl.DataFrame({
        "date": pl.datetime_range(start_h, end_h - interval_h, interval_h, eager=True),
        "location_Zst": zst,
        "location_name": name,
        "european_aqi": hourly.Variables(0).ValuesAsNumpy(),
        "pm10": hourly.Variables(1).ValuesAsNumpy(),
        "pm2_5": hourly.Variables(2).ValuesAsNumpy(),
        "nitrogen_dioxide": hourly.Variables(3).ValuesAsNumpy(),
        "carbon_monoxide": hourly.Variables(4).ValuesAsNumpy(),
        "ozone": hourly.Variables(5).ValuesAsNumpy(),
        "sulphur_dioxide": hourly.Variables(6).ValuesAsNumpy(),
        "carbon_dioxide": hourly.Variables(7).ValuesAsNumpy(),
    })
    
    all_dataframes.append(hourly_dataframe)

final_dataframe = pl.concat(all_dataframes)

final_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)

In [ ]:
#----Same script only for specific coordinates (kiel):-----   

In [6]:
import openmeteo_requests
import polars as pl
import requests_cache
import os
from retry_requests import retry
from datetime import datetime, timedelta, timezone

BASE_DIR   = "../../data"

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)



# URL and parameters remain the same
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": 54.3213,
	"longitude": 10.1349,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "hourly": ["european_aqi", "pm10", "pm2_5", "nitrogen_dioxide", "carbon_monoxide", "ozone", "sulphur_dioxide", "carbon_dioxide"]
}
response = openmeteo.weather_api(url, params=params)

In [7]:
# Hourly Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "AirQuality", "air_quality_kiel.csv")

all_dataframes = []
# Process first location
# Process location

# Get name
name = "Kiel"
  
# --- Process Hourly Data ---
hourly = response[0].Hourly()
  
#  calculate the range using datetime objects
start_h = datetime.fromtimestamp(hourly.Time(), tz=timezone.utc)
end_h = datetime.fromtimestamp(hourly.TimeEnd(), tz=timezone.utc)
 
interval_h = timedelta(seconds=hourly.Interval())
  
hourly_dataframe = pl.DataFrame({
   "date": pl.datetime_range(start_h, end_h - interval_h, interval_h, eager=True),
   "location_name": name,
   "european_aqi": hourly.Variables(0).ValuesAsNumpy(),
   "pm10": hourly.Variables(1).ValuesAsNumpy(),
   "pm2_5": hourly.Variables(2).ValuesAsNumpy(),
   "nitrogen_dioxide": hourly.Variables(3).ValuesAsNumpy(),
   "carbon_monoxide": hourly.Variables(4).ValuesAsNumpy(),
   "ozone": hourly.Variables(5).ValuesAsNumpy(),
   "sulphur_dioxide": hourly.Variables(6).ValuesAsNumpy(),
   "carbon_dioxide": hourly.Variables(7).ValuesAsNumpy(),
   })
    
all_dataframes.append(hourly_dataframe)
    

final_dataframe = pl.concat(all_dataframes)

final_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)